<div align="center">
  <img src="https://capsule-render.vercel.app/api?type=waving&color=0:0f2027,50:203a43,100:2c5364&height=200&section=header&text=Deep%Learning%20Projet&fontSize=50&fontColor=ffffff&animation=fadeIn&fontAlignY=38&desc=Fall%20Semester%202026/2027&descAlignY=58&descSize=18" width="100%" alt="banner"/>
</div>

In [2]:
import pandas as pd
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.model_selection import train_test_split, StratifiedKFold

In [3]:
labels_df = pd.read_csv("../Data/labels_with_hashes.csv")

In [4]:
labels_df.head()

,filename,class_id,description,hash,is_duplicate,n_labels
0,image_000001.JPEG,n03042490,cliff dwelling,4ea329c01903cc77177c0b62d6450784,False,1
1,image_000002.JPEG,n02793495,barn,d2e5e3766f67841827c73fdd234d69e6,False,1
2,image_000003.JPEG,n02793495,barn,441d7c58b3d4abaa7b1644340315dac1,False,1
3,image_000004.JPEG,n02793495,barn,fd0ba89ab10f1326d13019a5b63c3f1a,False,1
4,image_000005.JPEG,n02793495,barn,5332763e5065dde58d0ac261b0c53504,False,1


## **Data Cleaning**

This section will focus on dealing with the previously identified duplicate images and confliting Labels and incorrect labeling.

#### **Duplicate Handling**

In [8]:
labels_df_duplicates = labels_df[labels_df["is_duplicate"]]

labels_df_duplicates

,filename,class_id,description,hash,is_duplicate,n_labels
593,image_000594.JPEG,n03891332,parking meter,37bfb8e5b32c68152bfc8d2c7a53ee0b,True,2
1090,image_001091.JPEG,n07749582,lemon,59f2a108dfe8ec55424f514f2ff96a1c,True,2
11907,image_011908.JPEG,n01950731,"sea slug, nudibranch",3a7055e30d47b66c38cf3872564d9ddc,True,2
13623,image_013624.JPEG,n07747607,orange,37bfb8e5b32c68152bfc8d2c7a53ee0b,True,2
15109,image_015110.JPEG,n03584254,iPod,55985fcb729f347a8edf056fa147fb07,True,2
27766,image_027767.JPEG,n03404251,fur coat,0f400b529da4df75a409602c4c5e7db8,True,2
30312,image_030313.JPEG,n01945685,slug,3a7055e30d47b66c38cf3872564d9ddc,True,2
33613,image_033614.JPEG,n01984695,"spiny lobster, langouste, rock lobster, crawfi...",eb57816265bf1eb296181540fe6c2e27,True,2
40382,image_040383.JPEG,n01983481,"American lobster, Northern lobster, Maine lobs...",eb57816265bf1eb296181540fe6c2e27,True,2
47291,image_047292.JPEG,n02988304,CD player,0f400b529da4df75a409602c4c5e7db8,True,2


From the EDA notebook, we concluded that:

- Images with index 27766 and 47291 are the same but have two very different labels. As the image with index 47291 has a label that does not make any sense we're going to **drop that row.**
- Images with index 593 and 13623 are the same but the image with index 593 has a label that does not make sense, so we're also going to **drop that row.**

#### **Incorrect Labeling**

## **Data Splitting**

Dividing the dataset in Train, Validation and Test set in order to avoid data leakage.

In [14]:
labels_df.groupby("class_id")["description"].nunique().max()

1

As one class_id corresponds to one only label/description, we are going to stratify on `class_id`.

In [15]:
train_val_df, test_df = train_test_split(labels_df, test_size=0.15, stratify=labels_df["class_id"], random_state=42)

train_df, val_df = train_test_split(train_val_df, test_size=0.15, stratify=train_val_df["class_id"], random_state=42)

## **Image Transformations**

As previously observed in the EDA notebook, all the images have the same size, so we do not need to resize the images. This sections will focus on the normalization, which consists of, for each pixel, change the [0,255] range to [0,1], and convert each image to a tensor.

## **Data Augmentation & Class Imbalance Handling**

Through transformations such as, rotations, we will increase the amount of training data. This is specially useful in this case, as it is a way to decrease class imbalance, by focusing on Class-Specific Data Agumentation, where we apply augmentation more frequently to the minority classes.